# FaceForensics++ — Train Face2Face Detector (Session 2)

**Model:** Xception  
**Manipulation:** Deepfakes  
**Compression:** c23  
**GPU:** Tesla T4 (Kaggle)  

---
Chạy từng cell theo thứ tự từ trên xuống dưới.

## Cell 1 — Kiểm tra GPU

In [ ]:
!nvidia-smi

## Cell 2 — Kiểm tra input datasets

In [ ]:
import os

print('=== /kaggle/input ===')
print(os.listdir('/kaggle/input'))

print('\n=== Tìm dataset FF++ ===')
for name in os.listdir('/kaggle/input'):
    path = f'/kaggle/input/{name}'
    contents = os.listdir(path)
    print(f'{path}: {contents[:5]}')

## Cell 3 — Đặt biến đường dẫn

⚠️ **Sửa `FFPP_SLUG` và `CODE_SLUG` cho đúng với tên dataset trên Kaggle của bạn!**

In [ ]:
# ============================================================
# SỬA DƯỚI ĐÂY NẾU KAGGLE ĐƯỜNG DẪN KHÁC
# ============================================================
BASE_INPUT = '/kaggle/input/datasets/votantainu'
FFPP_SLUG = 'ffpp-c23-full'       # slug của dataset FF++ data
CODE_SLUG  = 'ffpp-training-code' # slug của dataset code repo
# ============================================================

FFPP_ROOT = f'{BASE_INPUT}/{FFPP_SLUG}'
CODE_ROOT = f'{BASE_INPUT}/{CODE_SLUG}'
WORK_DIR  = '/kaggle/working/ffpp'

print(f'FF++ data  : {FFPP_ROOT}')
print(f'Code input : {CODE_ROOT}')
print(f'Work dir   : {WORK_DIR}')

## Cell 4 — Cài dependencies

In [ ]:
!pip install -q timm facenet-pytorch scikit-learn PyYAML tqdm pandas matplotlib Pillow
print('Done installing dependencies.')

## Cell 5 — Copy code vào thư mục có thể ghi

In [ ]:
import shutil, os

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

shutil.copytree(CODE_ROOT, WORK_DIR)
print(f'Code copied to {WORK_DIR}')
print(os.listdir(WORK_DIR))

## Cell 5.5 — Tối ưu hóa tốc độ trích xuất khuôn mặt (Tăng tốc ~30 lần) ⚡

Mặc định, bộ nạp dữ liệu sẽ chạy MTCNN trên tất cả 32 frames của video rồi mới chọn ra 1 frame. Cell này sẽ patch code để dừng ngay khi tìm thấy khuôn mặt đầu tiên, giúp giảm thời gian train mỗi epoch từ **2 tiếng** xuống còn **~5 phút**.

In [ ]:
import re
path = f'{WORK_DIR}/datasets/ff_dataset.py'
code = open(path, encoding='utf-8').read()

old_loop = """        crops = []
        for frame_idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            success, frame = cap.read()
            if not success:
                continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            crop = self._extract_face(frame)
            if crop is not None:
                crops.append(crop)
        cap.release()

        if not crops:
            fallback_shape = (3, self.detector.config.output_size, self.detector.config.output_size)
            return torch.zeros(fallback_shape, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

        return crops[len(crops) // 2], torch.tensor(label, dtype=torch.long)"""

new_loop = """        crop = None
        for frame_idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            success, frame = cap.read()
            if not success:
                continue
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            detected_crop = self._extract_face(frame)
            if detected_crop is not None:
                crop = detected_crop
                break
        cap.release()

        if crop is None:
            fallback_shape = (3, self.detector.config.output_size, self.detector.config.output_size)
            return torch.zeros(fallback_shape, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

        return crop, torch.tensor(label, dtype=torch.long)"""

if old_loop in code:
    code = code.replace(old_loop, new_loop)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(code)
    print('✅ Đã tối ưu hóa ff_dataset.py thành công! Tốc độ sẽ tăng ~30x.')
else:
    print('⚠️ Không tìm thấy đoạn code cũ (có thể file đã được tối ưu hóa từ trước).')

## Cell 6 — Kiểm tra cấu trúc dataset FF++

In [ ]:
from pathlib import Path

checks = [
    f'{FFPP_ROOT}/splits/train.json',
    f'{FFPP_ROOT}/splits/val.json',
    f'{FFPP_ROOT}/splits/test.json',
    f'{FFPP_ROOT}/original_sequences/youtube/c23/videos',
    f'{FFPP_ROOT}/manipulated_sequences/Deepfakes/c23/videos',
]

all_ok = True
for p in checks:
    exists = Path(p).exists()
    icon = '✅' if exists else '❌'
    print(f'{icon} {p}')
    if not exists:
        all_ok = False

if all_ok:
    # Đếm video
    orig_count = len(list(Path(f'{FFPP_ROOT}/original_sequences/youtube/c23/videos').glob('*.mp4')))
    fake_count = len(list(Path(f'{FFPP_ROOT}/manipulated_sequences/Deepfakes/c23/videos').glob('*.mp4')))
    print(f'\nOriginal videos : {orig_count}')
    print(f'Deepfakes videos: {fake_count}')
    print('\n✅ Dataset OK — sẵn sàng train!')
else:
    print('\n❌ Thiếu file/thư mục. Kiểm tra lại FFPP_SLUG hoặc cấu trúc dataset.')

## Cell 7 — Tạo config Kaggle

Nếu `batch_size: 16` bị OOM thì giảm xuống `8`.

In [ ]:
from pathlib import Path

# Điều chỉnh nếu cần
BATCH_SIZE = 16   # Giảm xuống 8 nếu OOM
NUM_EPOCHS = 10
NUM_FRAMES = 32

config_content = f"""\
model_name: xception
pretrained: true
num_classes: 2
dropout: 0.5
input_size: 299

data_root: {FFPP_ROOT}
manipulation: Face2Face
compression: c23
batch_size: {BATCH_SIZE}
num_epochs: {NUM_EPOCHS}
num_frames_train: {NUM_FRAMES}
num_frames_eval: {NUM_FRAMES}
lr: 0.0002
weight_decay: 0.0
scheduler: reduce_on_plateau
loss_name: cross_entropy
label_smoothing: 0.0
num_workers: 0
checkpoint_dir: /kaggle/working/checkpoints
early_stopping_patience: 5
seed: 42
use_video_dataset: true
detector_backend: mtcnn
device: cuda
"""

cfg_path = Path(f'{WORK_DIR}/configs/kaggle_run.yaml')
cfg_path.write_text(config_content, encoding='utf-8')

print('Config saved to:', cfg_path)
print('---')
print(cfg_path.read_text())

## Cell 8 — Kiểm tra split files

Nếu splits thiếu, cell này sẽ tự download từ GitHub official.

In [ ]:
import json, urllib.request
from pathlib import Path

# splits_dir nằm trong /kaggle/input → read-only → dùng working dir trực tiếp
splits_work = Path('/kaggle/working/splits')
splits_work.mkdir(parents=True, exist_ok=True)

SPLIT_URLS = {
    'train.json': 'https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/train.json',
    'val.json'  : 'https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/val.json',
    'test.json' : 'https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/test.json',
}

for fname, url in SPLIT_URLS.items():
    target_work  = splits_work / fname
    
    # Thử tìm trong input trước, nếu không có sẽ tự tải từ GitHub
    target_input = Path(FFPP_ROOT) / 'splits' / fname
    if target_input.exists():
        print(f'✅ {fname} exists in dataset')
        import shutil
        shutil.copy(target_input, target_work)
    else:
        print(f'⬇️  Downloading {fname} from GitHub...')
        urllib.request.urlretrieve(url, target_work)
        print(f'✅ {fname} downloaded')
    
    # Verify
    with open(target_work) as f:
        data = json.load(f)
    print(f'   {fname}: {len(data)} entries\n')

print('Splits ready at:', splits_work)

## Cell 9 — Cập nhật config nếu splits ở working dir

In [ ]:
import shutil
from pathlib import Path

# Nếu splits nằm trong dataset input → OK, data_root đã đúng
# Nếu splits phải download → cần symlink hoặc copy vào working
# Cách an toàn nhất: tạo merged data dir

ffpp_input = Path(FFPP_ROOT)
ffpp_work_data = Path('/kaggle/working/ffpp_data')
ffpp_work_data.mkdir(parents=True, exist_ok=True)

# Symlink các thư mục lớn (không copy để tiết kiệm dung lượng)
for subdir in ['original_sequences', 'manipulated_sequences']:
    src = ffpp_input / subdir
    dst = ffpp_work_data / subdir
    if src.exists() and not dst.exists():
        dst.symlink_to(src)
        print(f'Symlinked: {dst} -> {src}')
    elif dst.exists():
        print(f'Already exists: {dst}')

# Copy splits vào đây
splits_dst = ffpp_work_data / 'splits'
if splits_dst.exists():
    shutil.rmtree(splits_dst)
shutil.copytree('/kaggle/working/splits', splits_dst)
print(f'Splits copied to: {splits_dst}')

# Cập nhật config để dùng path mới
DATA_ROOT_FINAL = str(ffpp_work_data)
print(f'\nFinal DATA_ROOT: {DATA_ROOT_FINAL}')

cfg_path = Path(f'{WORK_DIR}/configs/kaggle_run.yaml')
text = cfg_path.read_text()
text = text.replace(FFPP_ROOT, DATA_ROOT_FINAL)
cfg_path.write_text(text)
print('Config updated!')

## Cell 10 — Chạy training! 🚀

Đây là cell chính. Sẽ mất khoảng **1–2 giờ** với T4 GPU.

Theo dõi log để xem `train_loss`, `val_acc` từng epoch.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'training.trainer',
    '--config', f'{WORK_DIR}/configs/kaggle_run.yaml',
    '--data-root', DATA_ROOT_FINAL,
    '--manipulation', 'Face2Face',
    '--compression', 'c23',
    '--device', 'cuda',
]

print('Running:', ' '.join(cmd))
print('=' * 60)

result = subprocess.run(cmd, cwd=WORK_DIR, capture_output=False)
print('\n' + '=' * 60)
print(f'Return code: {result.returncode}')

## Cell 11 — Kiểm tra checkpoints

In [ ]:
from pathlib import Path
import os

ckpt_dir = Path('/kaggle/working/checkpoints')

if ckpt_dir.exists():
    files = list(ckpt_dir.iterdir())
    if files:
        print('Checkpoints saved:')
        for f in files:
            size_mb = f.stat().st_size / 1024 / 1024
            print(f'  {f.name}  ({size_mb:.1f} MB)')
    else:
        print('❌ Checkpoint dir rỗng — training chưa chạy hoặc bị lỗi')
else:
    print('❌ Checkpoint dir không tồn tại')

## Cell 12 — Copy best checkpoint ra /kaggle/working để download

In [ ]:
import shutil
from pathlib import Path

best = Path('/kaggle/working/checkpoints/best_Face2Face_c23.pth')
dest = Path('/kaggle/working/best_Face2Face_c23.pth')

if best.exists():
    shutil.copy(best, dest)
    size_mb = dest.stat().st_size / 1024 / 1024
    print(f'✅ Saved: {dest}  ({size_mb:.1f} MB)')
    print('Vào Output tab của Kaggle notebook để download file này.')
else:
    print('❌ best_Face2Face_c23.pth không tồn tại')
    print('Kiểm tra lại Cell 11 — training có thể chưa hoàn thành.')

## Cell 13 — (Tùy chọn) Vẽ training history

In [ ]:
# Cell này chỉ chạy được nếu trainer in ra JSON mỗi epoch
# Nếu muốn vẽ, cần bắt stdout từ Cell 10
# Đây là placeholder — bỏ qua nếu không cần
print('Xem log output ở Cell 10 để theo dõi train_loss và val_acc từng epoch.')

---
## Sau khi train xong

1. Download `best_Face2Face_c23.pth` từ **Output tab** của Kaggle
2. Đặt file vào máy local tại: `cuoi_ki_DL/checkpoints/best_Face2Face_c23.pth`
3. Chạy local inference:

```bash
python local_inference.py checkpoints/best_Face2Face_c23.pth path/to/video.mp4
```

---
## Troubleshooting

| Lỗi | Cách xử lý |
|-----|------------|
| CUDA OOM | Giảm `BATCH_SIZE = 8` ở Cell 7, chạy lại từ Cell 7 |
| No samples found | Kiểm tra `FFPP_SLUG` đúng chưa ở Cell 3 |
| MTCNN error | Đổi `detector_backend: haar` trong Cell 7 |
| Session timeout | Giảm `NUM_EPOCHS = 5` và chạy 2 session |


## Cell 14 — Đánh giá chi tiết trên tập TEST (F1-Score, Accuracy, Precision, Recall) 📈

Cell này sẽ chạy đánh giá mô hình tốt nhất (`best_*.pth`) vừa train xong trên tập kiểm thử độc lập (Test Set) của video-level, hiển thị chi tiết các chỉ số F1, Accuracy, Precision, Recall và Confusion Matrix cho báo cáo của bạn.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

# Tự động import từ codebase đã copy
import sys
sys.path.append('/kaggle/working/ffpp')
from datasets.ff_dataset import FaceForensicsVideoDataset
from datasets.transforms import build_transforms
from inference.predict_image import load_model_from_checkpoint
from training.trainer import resolve_device

# Xác định tên manipulation của notebook này
manipulation = "Face2Face"

best_ckpt_path = Path('/kaggle/working/checkpoints') / f'best_{manipulation}_c23.pth'

if best_ckpt_path.exists():
    device = resolve_device('cuda')
    model, checkpoint = load_model_from_checkpoint(best_ckpt_path, 'xception', device)
    transforms = build_transforms('xception')
    
    test_dataset = FaceForensicsVideoDataset(
        data_root=DATA_ROOT_FINAL,
        split="test",
        manipulation=manipulation,
        compression="c23",
        num_frames=32,
        transform=transforms["test"],
        detector_backend="mtcnn",
        detector_device="cuda"
    )
    
    # Đặt num_workers=0 để tránh lỗi multiprocessing CUDA
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)
    
    probs = []
    labels = []
    
    print(f"🔄 Đang đánh giá mô hình {manipulation} trên tập TEST ({len(test_dataset)} videos)...")
    with torch.no_grad():
        for i, (images, batch_labels) in enumerate(test_loader):
            images = images.to(device)
            logits = model(images)
            batch_probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy().tolist()
            probs.extend(batch_probs)
            labels.extend(batch_labels.numpy().tolist())
            if (i+1) % 5 == 0 or (i+1) == len(test_loader):
                print(f"   Đã quét: {min((i+1)*16, len(test_dataset))}/{len(test_dataset)} videos")
                
    y_true = np.array(labels)
    y_prob = np.array(probs)
    y_pred = (y_prob > 0.5).astype(int)
    
    acc = accuracy_score(y_true, y_pred) * 100.0
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except:
        auc = float('nan')
    cm = confusion_matrix(y_true, y_pred).tolist()
    
    print("\n" + "="*50)
    print("📈 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (VIDEO-LEVEL)")
    print("="*50)
    print(f"| Chỉ số (Metric) | Giá trị (Value) |")
    print(f"| :--- | :--- |")
    print(f"| **Accuracy (Độ chính xác)** | {acc:.2f}% |")
    print(f"| **F1-Score** | {f1:.4f} |")
    print(f"| **Precision (Độ chính xác dự báo)** | {precision:.4f} |")
    print(f"| **Recall (Độ phủ)** | {recall:.4f} |")
    print(f"| **AUC-ROC** | {auc:.4f} |")
    print("-"*50)
    print(f"Confusion Matrix: TN={cm[0][0]}, FP={cm[0][1]}, FN={cm[1][0]}, TP={cm[1][1]}")
    print("="*50 + "\n")
else:
    print(f"❌ Không tìm thấy file checkpoint: {best_ckpt_path}")
